# Nhóm 3 — MNIST CNN-LSTM trên Kaggle
Dùng chung lõi `src/` và runner; seed 42, thiết bị CUDA tự chọn nếu runtime cấp GPU.

In [ ]:
from pathlib import Path
import os,sys,torch
PROJECT_ROOT=Path('/kaggle/working/BTL03'); INPUT_ROOT=Path('/kaggle/input'); WORK_ROOT=Path('/kaggle/working')
if not (PROJECT_ROOT/'scripts/run_group3_cnn_lstm.py').is_file(): raise FileNotFoundError('Checkout BTL03 vào /kaggle/working/BTL03.')
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Kaggle datasets:',[p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else [])

## Ánh xạ MNIST từ Kaggle Input (nếu có)
Có thể đính kèm cache `train_images.npy`, `train_labels.npy`, `test_images.npy`, `test_labels.npy`. Nếu chưa có, runner gọi downloader MNIST IDX chính thức.

In [ ]:
import shutil
for basename in ['train_images.npy','train_labels.npy','test_images.npy','test_labels.npy']:
    matches=list(INPUT_ROOT.rglob(basename)) if INPUT_ROOT.exists() else []
    if matches:
        target=PROJECT_ROOT/'data/raw/mnist'/basename; target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists(): shutil.copy2(matches[0],target)
!python scripts/run_group3_cnn_lstm.py --dataset-only

In [ ]:
!python scripts/run_group3_cnn_lstm.py

In [ ]:
import json
from IPython.display import display,Image
metrics=json.loads((PROJECT_ROOT/'outputs/figures/t0207_cnn_lstm_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics,ensure_ascii=False,indent=2))
display(Image(filename=str(PROJECT_ROOT/metrics['figure'])))

## Nén artifacts vào `/kaggle/working/group3_cnn_lstm_artifacts.zip`
Không đưa dữ liệu raw vào ZIP mặc định; chỉ gồm checkpoints, figures, CSV và TensorBoard logs.

In [ ]:
import shutil
artifact=WORK_ROOT/'group3_cnn_lstm_artifacts'
artifact.mkdir(exist_ok=True)
for relative in ['outputs/checkpoints/03_cnn_lstm','outputs/figures/t0206_seq_mnist_samples.png','outputs/figures/t0207_cnn_lstm_results.png','outputs/figures/t0207_cnn_lstm_metrics.json','experiments/csv_logs','experiments/tensorboard/Group3_CNN_LSTM']:
    source=PROJECT_ROOT/relative
    if source.exists():
        target=artifact/relative.replace('/','_')
        if source.is_dir(): shutil.copytree(source,target,dirs_exist_ok=True)
        else: shutil.copy2(source,target)
zip_path=shutil.make_archive(str(WORK_ROOT/'group3_cnn_lstm_artifacts'),'zip',artifact)
print('Artifact ZIP:',zip_path)